# Exercice 3 - Remplacez la Q-table par un cerveau neuronal (DQN)

## 🎯 Objectif pédagogique

Dans cet exercice, nous allons passer de la **Q-table** (exercice 2) à un **réseau de neurones** qui remplace la table.

**Problème avec la Q-table :**
- FrozenLake : 16 états × 4 actions = 64 valeurs ✅
- CartPole : états **continus** (positions, vitesses) = ∞ états ❌

**Solution :** Le réseau de neurones **approxime** la fonction Q, même pour des espaces continus.

```
Q-table :     État → [Q1, Q2, Q3, Q4]  (tableau)
DQN :         État → [Q1, Q2, Q3, Q4]  (réseau de neurones)
```

## 🕹️ Contexte : Qu'est-ce qu'un DQN ?

Un **Deep Q-Network (DQN)** est un réseau de neurones qui estime la fonction Q :

```
Entrée (observation) → [Couche 1] → [Couche 2] → Sortie [Q(gauche), Q(droite)]
   4 valeurs          64 neurones     64 neurones     2 valeurs
```

**Avantages du DQN :**
- Gère les espaces **continus** (CartPole, LunarLander)
- **Généralise** : peut estimer Q pour des états jamais vus

**Inconvénients :**
- Plus complexe à implémenter
- Plus difficile à entraîner (instabilité)

## 1. Import des bibliothèques

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random

**Explication :**
- `torch` :框架 d'apprentissage profond
- `torch.nn` :模块 pour construire les réseaux
- `torch.optim` : optimiseurs (Adam, etc.)
- `deque` : buffer pour le replay memory

## 2. Partie 1 : Implémentation du DQN avec PyTorch

### Le réseau de neurones (Q-Network)

In [ ]:
class QNetwork(nn.Module):
    """Réseau de neurones qui approxime la fonction Q."""
    
    def __init__(self, state_size, action_size, hidden_size=64):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, action_size)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

**💡 Explication :**

```
Entrée (4) → fc1 (64) → ReLU → fc2 (64) → ReLU → fc3 (2) → Sortie
```

| Couche | Rôle |
|--------|------|
| `fc1` | Transforme l'observation en vecteur de 64 |
| `fc2` | Traitement intermédiaire |
| `fc3` | Produce une valeur Q par action |
| `ReLU` | Fonction d'activation (permet la non-linéarité) |

### Le Replay Buffer

In [ ]:
class ReplayBuffer:
    """Mémoire pour stocker et échantillonner les expériences."""
    
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)
    
    def __len__(self):
        return len(self.buffer)

**💡 Pourquoi un Replay Buffer ?**

Sans replay buffer :
- L'agent apprend d'une expérience à la fois
- Les expériences consécutives sont **corrélées** (très similaires)

Avec replay buffer :
- On stocke les expériences dans une mémoire
- On échantillonne **aléatoirement** pour briser la corrélation
- Cela stabilise l'entraînement

### 📝 Synthèse de la Partie 1

**Ce qu'on a appris :**
- ✅ Le Q-Network remplace la Q-table
- ✅ Le Replay Buffer stocke les expériences pour un apprentissage stable

**Concepts clés :**
- **Q-Network** = approximateur de la fonction Q
- **Replay Buffer** = mémoire des expériences (déqueue avec maxlen)

## 3. Partie 2 : L'algorithme DQN (avec Target Network)

### Pourquoi un Target Network ?

**Problème d'instabilité :**

Dans la formule de Bellman :
`Q(s,a) = r + γ × max(Q(s',a'))`

- La **cible** (`r + γ × max(Q(s',a'))`) dépend de Q
- Si Q change, la cible change
- C'est une cible mouvante → instabilité

**Solution :** Utiliser 2 réseaux :
- **Policy Network** : celui qu'on entraîne
- **Target Network** : dont on calcule la cible (mis à jour moins souvent)

### L'algorithme DQN complet

In [ ]:
def train_dqn(env, n_episodes=500, batch_size=64, gamma=0.99, lr=1e-3, epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995):
    """Entraîne un agent DQN sur CartPole-v1."""
    
    # Paramètres
    state_size = env.observation_space.shape[0]  # 4
    action_size = env.action_space.n              # 2
    
    # Réseaux de neurones
    policy_net = QNetwork(state_size, action_size)
    target_net = QNetwork(state_size, action_size)
    target_net.load_state_dict(policy_net.state_dict())
    target_net.eval()
    
    optimizer = optim.Adam(policy_net.parameters(), lr=lr)
    memory = ReplayBuffer(capacity=10000)
    
    epsilon = epsilon_start
    rewards_history = []
    
    for episode in range(n_episodes):
        state, info = env.reset()
        total_reward = 0
        terminated = truncated = False
        
        while not (terminated or truncated):
            # 1. Choix de l'action (epsilon-greedy)
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action = policy_net(torch.FloatTensor(state)).argmax().item()
            
            # 2. Exécuter l'action
            next_state, reward, terminated, truncated, info = env.step(action)
            
            # 3. Stocker dans le replay buffer
            memory.push(state, action, reward, next_state, terminated or truncated)
            
            state = next_state
            total_reward += reward
            
            # 4. Entraîner si assez d'expériences
            if len(memory) >= batch_size:
                train_step(policy_net, target_net, optimizer, memory, batch_size, gamma)
        
        # 5. Mettre à jour epsilon
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        rewards_history.append(total_reward)
        
        # 6. Mettre à jour le target network tous les 10 épisodes
        if episode % 10 == 0:
            target_net.load_state_dict(policy_net.state_dict())
        
        if (episode + 1) % 50 == 0:
            print(f"Episode {episode+1}/{n_episodes} - Reward moyen: {np.mean(rewards_history[-50:]):.1f} - Epsilon: {epsilon:.3f}")
    
    return policy_net, rewards_history


def train_step(policy_net, target_net, optimizer, memory, batch_size, gamma):
    """Une étape d'entraînement du DQN."""
    
    # 1. Échantillonner un batch
    batch = memory.sample(batch_size)
    states, actions, rewards, next_states, dones = zip(*batch)
    
    # Convertir en tensors
    states = torch.FloatTensor(np.array(states))
    actions = torch.LongTensor(actions)
    rewards = torch.FloatTensor(rewards)
    next_states = torch.FloatTensor(np.array(next_states))
    dones = torch.FloatTensor(dones)
    
    # 2. Calculer Q(s, a) - valeur actuelle
    current_q = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze()
    
    # 3. Calculer Q(s', a') target avec le target network
    with torch.no_grad():
        next_q = target_net(next_states).max(1)[0]
    
    # 4. Cible = r + gamma * max(Q(s', a')) * (1 - done)
    target_q = rewards + gamma * next_q * (1 - dones)
    
    # 5. Loss = MSE entre Q actuel et Q cible
    loss = nn.MSELoss()(current_q, target_q)
    
    # 6. Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

**💡 Explication détaillée de l'entraînement :**

```
1. Échantillonner un batch d'expériences
2. Calculer Q(s, a) avec policy_net
3. Calculer max(Q(s', a')) avec target_net
4. Cible = r + gamma * max(Q(s', a')) * (1 - done)
5. Loss = (Q(s,a) - cible)²
6. Mettre à jour policy_net pour réduire la loss
```

**Question :** Pourquoi utiliser `gather(1, actions.unsqueeze(1))` ?

→ Pour extraire uniquement la valeur Q de l'action qui a été réellement jouée.

### Lancer l'entraînement

In [ ]:
# Créer l'environnement
env = gym.make("CartPole-v1")

print("Lancement de l'entraînement DQN...")
print("(Cela peut prendre quelques minutes)")
print("-" * 50)

policy_net, rewards_history = train_dqn(env, n_episodes=300)

env.close()

### Visualiser les résultats

In [ ]:
plt.figure(figsize=(12, 5))

# Graphe 1 : Reward par épisode
plt.subplot(1, 2, 1)
plt.plot(rewards_history, alpha=0.3)

# Moyenne mobile
window = 20
moving_avg = np.convolve(rewards_history, np.ones(window)/window, mode='valid')
plt.plot(moving_avg, color='red', linewidth=2, label=f'Moyenne mobile ({window})')

plt.xlabel('Épisode')
plt.ylabel('Reward')
plt.title('Apprentissage DQN')
plt.legend()
plt.grid(alpha=0.3)

# Graphe 2 : Comparaison avec agent aléatoire
plt.subplot(1, 2, 2)
random_rewards = [np.mean([10 + random.randint(-10, 10) for _ in range(50)]) for _ in range(6)]
dqn_rewards = [np.mean(rewards_history[i*50:(i+1)*50]) for i in range(6)]

x = range(len(random_rewards))
plt.bar([i-0.2 for i in x], random_rewards, 0.4, label='Aléatoire (~20)', color='gray', alpha=0.7)
plt.bar([i+0.2 for i in x], dqn_rewards, 0.4, label='DQN', color='blue', alpha=0.7)

plt.xlabel('Blocs de 50 épisodes')
plt.ylabel('Reward moyen')
plt.title('Comparaison DQN vs Aléatoire')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n=== Résumé ===")
print(f"Reward moyen (50 derniers) : {np.mean(rewards_history[-50:]):.1f}")
print(f"Reward max : {np.max(rewards_history)}")

### 📝 Synthèse de la Partie 2

**Ce qu'on a appris :**
- ✅ L'architecture DQN avec policy et target networks
- ✅ Le Replay Buffer pour stabiliser l'entraînement
- ✅ La stratégie epsilon-greedy

**Concepts clés :**
- **Target Network** : réseau qui calcule la cible (mis à jour périodiquement)
- **MSE Loss** : erreur quadratique entre Q prédit et Q cible
- **Backpropagation** : mise à jour des poids du réseau

## 4. Partie 3 : DQN avec Stable-Baselines3 (la solution professionnelle)

Maintenant que nous comprenons le DQN "from scratch", voyons comment **les professionnels** font :

**Stable-Baselines3** fournit des implémentations optimisées de nombreux algorithmes RL.

In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy

### Entraîner avec SB3

In [ ]:
# Créer l'environnement
env = gym.make("CartPole-v1")

# Créer le modèle DQN
model = DQN(
    policy='MlpPolicy',
    env=env,
    learning_rate=1e-3,
    buffer_size=10000,
    learning_starts=1000,
    target_update_interval=10,
    gamma=0.99,
    epsilon_decay=0.995,
    verbose=1
)

print("Entraînement DQN avec Stable-Baselines3...")
print("-" * 50)

# Entraîner
model.learn(total_timesteps=25000, progress_bar=True)

print("\nEntraînement terminé !")

**💡 Explication des paramètres :**

| Paramètre | Description |
|----------|-------------|
| `MlpPolicy` | Réseau de neurones simple (Multi-Layer Perceptron) |
| `buffer_size` | Taille du Replay Buffer |
| `learning_starts` | Étapes avant de commencer l'entraînement |
| `target_update_interval` | Fréquence de mise à jour du Target Network |
| `epsilon_decay` | Décroissance de l'exploration |

### Évaluer le modèle

In [ ]:
# Évaluer le modèle
print("Évaluation du modèle...")
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=100)

print(f"\n=== Résultats ===")
print(f"Reward moyen : {mean_reward:.1f} ± {std_reward:.1f}")
print(f"Reward max théorique : 500")

**Question :** Pourquoi `total_timesteps=25000` et non pas 25000 épisodes ?

→ `total_timesteps` = nombre d'**interactions** avec l'environnement, pas d'épisodes. Un épisode dure en moyenne ~20-50 steps, donc 25000 steps ≈ 500-1000 épisodes.

### Sauvegarder et charger le modèle

In [ ]:
# Sauvegarder
model.save("dqn_cartpole")
print("Modèle sauvegardé : dqn_cartpole.zip")

# Charger (si nécessaire)
# loaded_model = DQN.load("dqn_cartpole")

# Fermer l'environnement
env.close()

### 📝 Synthèse de la Partie 3

**Ce qu'on a appris :**
- ✅ Stable-Baselines3 simplifie drastiquement l'implémentation
- ✅ Quelques lignes suffisent pour entraîner un agent performant
- ✅ Comment sauvegarder et charger un modèle

**Comparaison :**

| Aspect | DQN from scratch | Stable-Baselines3 |
|--------|------------------|------------------|
| Lignes de code | ~100 | ~10 |
| Contrôle | Total | Limité |
| Rapidité de développement | Long | Court |
| Flexibilité | Haute | Moyenne |

## 5. Conclusion

### Ce qu'on a accompli

| Compétence | Status |
|-----------|--------|
| Implémenter un Q-Network avec PyTorch | ✅ |
| Comprendre le Replay Buffer | ✅ |
| Comprendre le Target Network | ✅ |
| Entraîner un DQN from scratch | ✅ |
| Utiliser Stable-Baselines3 | ✅ |

### Concepts clés du DQN

```
DQN = Q-Learning + Réseau de neurones + Replay Buffer + Target Network
```

1. **Q-Network** : remplace la Q-table pour gérer les espaces continus
2. **Replay Buffer** : stocke les expériences pour un apprentissage stable
3. **Target Network** : fournit une cible stable pour l'entraînement
4. **Epsilon-greedy** : équilibre exploration/exploitation

### Prochaines étapes

- **Mission :** Appliquer au LunarLander (score > 200)
- Implémenter PPO (autre algorithme populaire)
- Explorer des environnements plus complexes